# Семинар 6. Pandas: типы, даты, пропуски, объединение таблиц

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-06.ipynb`, например `2MP9-Ivanov-Ivan-seminar-06.ipynb`.
- Ноутбук читает файлы `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv` из репозитория курса: путь в ячейке задан от папки `notebooks/`, поэтому папки `notebooks/` и `datasets/` должны лежать рядом, как в репозитории. Описание данных — в `datasets/legal/README.md`.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-06-tasks.md` с формулировками и ноутбук `seminar-06-tasks.ipynb` с заготовками и тестами; они решаются на тех же данных, но по Московской области. Ответы вводятся в тест Moodle «Семинар 6».
- Шпаргалка по функциям: `reference/pandas.md`.
- Следующее занятие — контрольная работа № 1; правила и пустой ноутбук для неё лежат в `notebooks/seminar-07-test-1.ipynb`.

## 1. Две таблицы и три вопроса

У кейса две таблицы. В таблице ДТП одна строка — одно происшествие, номер ДТП стоит в индексе. В таблице участников одна строка — один человек: водитель, пассажир, пешеход; своего номера у участника нет, а столбец `dtp_id` говорит, в каком ДТП он участвовал. До сих пор работала только первая таблица, и работала как готовая: год брался срезом строки, пропуски обходились стороной.

In [ ]:
import numpy as np
import pandas as pd

dtp = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id')      # ДТП Брянской области
people = pd.read_csv('../datasets/legal/dtp-bryansk-participants.csv')      # участники этих ДТП
dtp.shape, people.shape                                                     # строки и столбцы таблиц

In [ ]:
people.head(3)                                                           # первые три участника

10 114 ДТП и 24 717 участников. Три вопроса, на которые таблицы в таком виде не отвечают:

1. В какие дни недели и в какие часы ДТП больше всего? Дата записана текстом, у текста нет дня недели.
2. Какой стаж у водителей, попавших в ДТП? В столбце стажа пропуски и значения, которые стажем быть не могут.
3. Кто погибает чаще: пешеход, сбитый днём, или пешеход, сбитый ночью? Роль и состояние здоровья лежат в одной таблице, освещение в другой.

Первому вопросу нужны настоящие даты, второму чистка данных, третьему обе таблицы вместе. Начинается всё с типов столбцов.

## 2. Типы

Тип столбца определяет, что с ним можно делать. `read_csv` выбирает тип сам по содержимому файла.

In [ ]:
people.dtypes                                                            # типы столбцов участников

Номер транспортного средства, стаж и год выпуска оказались дробными, `float64`, хотя по смыслу это целые числа. Причина в пропусках: у пешехода нет ни машины, ни стажа, ячейка в файле пустая, а пустое значение в обычный целый тип не помещается. Перевод в целый тип методом `astype(int)` на таком столбце завершается ошибкой, в следующей ячейке она намеренная.

In [ ]:
people['vehicle_year'].astype(int)                                       # целый тип и пропуски: ошибка

Сообщение об ошибке само подсказывает выход: у pandas есть целый тип, который умеет хранить пропуски, `Int64` с прописной буквы. Пропуск в таком столбце печатается как `<NA>`.

In [ ]:
people['vehicle_year'].astype('Int64').tail(4)                           # годы выпуска целыми числами

`astype` переводит и в другие типы: `astype(float)`, `astype(str)`. Если числа прочитались текстом из-за мусора в файле, помогает функция `pd.to_numeric` с параметром `errors='coerce'`: всё, что числом не является, она заменяет пропуском. Для дат есть своя функция.

## 3. Даты

Столбец `datetime` имеет тип `str`: для pandas это текст, у которого нет ни года, ни дня недели. Свойства даты доступны через `dt`, но только у настоящих дат; ошибка в следующей ячейке намеренная.

In [ ]:
dtp['datetime'].dt.year                                                  # dt у текста: ошибка

Функция `pd.to_datetime` превращает текст в даты. Результат записывается обратно в тот же столбец: тип становится `datetime64`, а таблица выглядит как прежде.

In [ ]:
dtp['datetime'] = pd.to_datetime(dtp['datetime'])                        # текст становится датой
dtp['datetime'].head(3)

Теперь у столбца есть свойства: `dt.year`, `dt.month`, `dt.day`, `dt.hour`, `dt.dayofweek`. День недели pandas считает с нуля: понедельник 0, воскресенье 6. Первый вопрос занятия решается двумя строками.

In [ ]:
dtp['datetime'].dt.dayofweek.value_counts().sort_index()                 # ДТП по дням недели

In [ ]:
dtp['datetime'].dt.day_name().value_counts().head(3)                     # три самых аварийных дня

In [ ]:
dtp['datetime'].dt.hour.value_counts().head()                            # пять самых аварийных часов

Больше всего ДТП в пятницу и субботу, по часам пик приходится на 18 часов, вечерний разъезд. `day_name` называет дни по-английски.

Период отбирается сравнением столбца дат со строкой вида `'2025-01-01'`: pandas сам превращает её в дату. Дата без времени означает полночь, поэтому правая граница берётся как следующий день со знаком «меньше». Нерабочие праздничные дни января по статье 112 Трудового кодекса РФ: новогодние каникулы 1–6 и 8 января и Рождество Христово 7 января, восемь дней подряд. ДТП за эти дни 2025 года:

In [ ]:
holidays = (dtp['datetime'] >= '2025-01-01') & (dtp['datetime'] < '2025-01-09')   # маска: праздничные дни января
holidays.sum()

Разность двух дат — промежуток времени, объект `Timedelta`; число полных дней в нём хранит свойство `days`. Одна дата создаётся как `pd.Timestamp('2025-01-01')`, промежуток как `pd.Timedelta(days=8)`, и дату с промежутком можно складывать.

In [ ]:
span = dtp['datetime'].max() - dtp['datetime'].min()                     # от первого ДТП до последнего
span, span.days

In [ ]:
pd.Timestamp('2025-01-01') + pd.Timedelta(days=8)                        # конец праздничных дней

Если заранее известно, в каком столбце файла даты, их можно прочитать датами сразу: параметр `parse_dates` функции `read_csv`.

In [ ]:
dated = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id',
                    parse_dates=['datetime'])                            # даты при чтении
dated['datetime'].head(2)

## 4. Пропуски

Пропуск в pandas — особое значение `NaN`, у целых типов с пропусками `<NA>`. Метод `isna` даёт маску пропусков; её сумма по столбцам показывает, где и сколько пусто, среднее — какая это доля.

In [ ]:
people.isna().sum()                                                      # пропуски по столбцам

In [ ]:
people['experience'].isna().mean().round(3)                              # доля пропусков стажа

Стаж не указан у 9 286 участников, больше трети таблицы. Прежде чем что-то делать с пропусками, надо понять, чьи они.

In [ ]:
people.loc[people['experience'].isna(), 'role'].value_counts().head(4)   # у кого стаж не указан

У пассажиров и пешеходов стажа нет по смыслу: пропуск означает «не применимо». У 228 водителей стаж должен быть, но в источнике его нет: пропуск означает «неизвестно». Это разные пропуски, и обходятся с ними по-разному.

Статистики pandas пропуски не учитывают: среднее считается по указанным значениям. `count` считает указанные значения, `len` — все строки.

In [ ]:
people['experience'].mean().round(1), people['experience'].count(), len(people)   # среднее, указано, всего

С пропусками делают одно из двух. Первое — заполнить: метод `fillna` подставляет значение вместо пропуска и возвращает новый столбец, который присваивается обратно. В таблице ДТП пропуск в `road_defect` означает, что недостатки дороги не установлены, и это можно записать словами.

In [ ]:
dtp['road_defect'] = dtp['road_defect'].fillna('не установлены')         # пропуск получает название
dtp['road_defect'].value_counts().head(3)

Второе — удалить строки с пропусками методом `dropna`. Без параметров он удаляет строку, если пропуск есть хотя бы в одном столбце, и это ловушка: из 24 717 участников остаётся 14 435.

In [ ]:
len(people.dropna()), len(people.dropna(subset=['experience']))          # строк после удаления

In [ ]:
people.dropna()['role'].value_counts()                                   # кто остался после dropna

После `dropna()` в таблице остались почти одни водители: пассажиры и пешеходы удалены, потому что у них по смыслу нет стажа или машины. Параметр `subset` ограничивает проверку названными столбцами. Сама таблица `people` при этом не изменилась: результат `dropna` никуда не присвоен.

## 5. Коды вместо значений и дубликаты

Пропуск виден сразу. Хуже значения, которые выглядят как данные, но ими не являются. Сводка стажа показывает наибольшее значение 99 лет.

In [ ]:
people['experience'].describe().round(1)                                 # сводка стажа

In [ ]:
people['experience'].value_counts().sort_index().tail(7)                 # самые большие значения стажа

Стаж 60–69 лет встречается единицы раз, значений от 70 до 94 нет вовсе, а 95, 96 и 99 встречаются сотнями. Судя по распределению, это не годы, а служебные коды источника. Что означает каждый из них, по таблице не установить, но стажем они не являются. Метод `replace` заменяет перечисленные значения на пропуск `np.nan`; результат записывается в новый столбец, сырые данные остаются в таблице. Заменять надо пропуском, а не нулём: ноль лет стажа — настоящее значение.

In [ ]:
people['experience_clean'] = people['experience'].replace([95, 96, 99], np.nan)   # коды становятся пропусками
people['experience'].mean().round(1), people['experience_clean'].mean().round(1)  # средний стаж до и после

Средний стаж был 23,3 года, стал 16,0: почти треть среднего составляли коды. Медиана изменилась меньше, с 15 до 13 лет. Ответ на второй вопрос занятия будет считаться по очищенному столбцу.

Такие же значения бывают у года выпуска: в четырёх строках он равен 1.

In [ ]:
people.loc[people['vehicle_year'] == 1, ['dtp_id', 'role', 'brand', 'vehicle_year']]   # год выпуска 1

Вторая проверка — повторы. `duplicated` даёт маску строк, полностью повторяющих одну из предыдущих.

In [ ]:
dtp.duplicated().sum(), people.duplicated().sum()                        # повторы в двух таблицах

In [ ]:
people[people.duplicated(keep=False)].head(6)                            # как выглядят повторы

В таблице ДТП повторов нет. В таблице участников 1 193 строки повторяют другие. Первые из них — шесть пассажиров одной машины в ДТП 133, четверо мужчин и две женщины, все ранены: своего номера у человека нет, и строки людей одного пола с одним исходом одинаковы. Ошибка это или разные люди, проверяется по таблице ДТП: в ней записано число участников происшествия.

In [ ]:
dtp.loc[133, 'participants_count'], (people['dtp_id'] == 133).sum()      # участников по двум таблицам

В таблице ДТП у происшествия 133 восемь участников, и строк в таблице участников тоже восемь: повторы — разные люди, и `drop_duplicates` удалил бы настоящих людей. Прежде чем удалять повторы, надо ответить, что такое строка таблицы.

Повторы бывают и полезны. Сведения о машине записаны у каждого, кто в ней ехал; если оставить только столбцы о машине и удалить повторы, из таблицы людей получится таблица машин.

In [ ]:
vehicles = people.dropna(subset=['vehicle_no'])[['dtp_id', 'vehicle_no', 'brand', 'vehicle_year']]   # участники в машинах
vehicles = vehicles.drop_duplicates(ignore_index=True)                   # одна строка на машину
len(vehicles)

## 6. Объединение таблиц

Третий вопрос требует поставить рядом с каждым участником сведения о его ДТП. Это делает `merge`: строки двух таблиц сопоставляются по ключу. Если ключ не назвать, pandas ищет столбцы с одинаковыми именами, а их здесь нет: ошибка в следующей ячейке намеренная.

In [ ]:
people.merge(dtp)                                                        # ключ не назван: ошибка

Ключ слева — столбец `dtp_id` таблицы участников, ключ справа — индекс таблицы ДТП: параметры `left_on` и `right_index`. Первая проверка после объединения — число строк: оно должно совпасть с числом участников, каждому нашлось его ДТП.

In [ ]:
both = people.merge(dtp, left_on='dtp_id', right_index=True)             # участник и его ДТП
both.shape

In [ ]:
both[['dtp_id', 'role', 'health', 'datetime', 'district', 'light']].head(3)   # столбцы из двух таблиц

Параметр `how` решает, что делать со строками, которым пары не нашлось. По умолчанию `how='inner'`: остаются только строки с парой. При `how='left'` остаются все строки левой таблицы, а столбцы правой у строк без пары заполняются пропусками. Разница видна, если справа взять только ДТП с погибшими.

In [ ]:
fatal = dtp[dtp['dead_count'] > 0]                                       # ДТП с погибшими
inner = people.merge(fatal, left_on='dtp_id', right_index=True)          # только участники этих ДТП
left = people.merge(fatal, left_on='dtp_id', right_index=True, how='left')   # все участники
len(inner), len(left), left['district'].isna().sum()

В ДТП с погибшими участвовали 3 589 человек, и только они остались при `how='inner'`. При `how='left'` строк столько же, сколько участников, 24 717, но у 21 128 из них столбцы ДТП пустые: их ДТП в правой таблице нет. После объединения всегда проверяются число строк и пропуски в присоединённых столбцах.

Когда таблицы не сопоставляются, а складываются друг под другом, нужен не `merge`, а `concat`. У таблиц ДТП двух областей одинаковые столбцы; чтобы после склейки знать, откуда строка, каждой заранее добавляется столбец с названием области. Доля ДТП с погибшими по областям за общие годы, 2022–2025:

In [ ]:
msk = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id',
                  parse_dates=['datetime'])                              # ДТП Московской области
bryansk = dtp.copy()                                                     # копия для склейки
bryansk['region'] = 'Брянская'
msk['region'] = 'Московская'
two = pd.concat([bryansk, msk])                                          # таблицы друг под другом
recent = two[two['datetime'].dt.year >= 2022]                            # общие годы
(recent['dead_count'] > 0).groupby(recent['region']).agg(['size', 'mean']).round(3)

Доли близки, 0,173 и 0,158, но сравнивать области по ним надо с оговоркой: учёт лёгких ДТП у регионов разный, как и у разных лет; в описании данных об этом сказано.

## 7. Мини-анализ и итоги

Первый вопрос, дни и часы, решён в разделе 3; к нему одно дополнение. Ночных ДТП мало, но они тяжелее: доля ДТП в часы с полуночи до шести утра и доля ДТП с погибшими в эти часы и в остальное время.

In [ ]:
night_hours = dtp['datetime'].dt.hour.between(0, 5)                      # маска: ночные часы
with_dead = dtp['dead_count'] > 0                                        # маска: ДТП с погибшими
night_hours.mean().round(3), with_dead[night_hours].mean().round(3), with_dead[~night_hours].mean().round(3)

На часы с полуночи до шести утра приходится 8,5 % ДТП, но погибшие есть почти в каждом четвёртом из них против каждого восьмого в остальное время.

Второй вопрос, стаж водителей: медиана очищенного стажа по тяжести ДТП. Таблица `both` собрана после чистки, поэтому очищенный столбец в ней есть.

In [ ]:
drivers = both[both['role'] == 'Водитель']                               # водители с их ДТП
drivers.groupby('severity')['experience_clean'].agg(['size', 'median'])  # водители и медиана стажа

Медиана стажа 12–14 лет, и в ДТП с погибшими она не ниже, чем в лёгких. Вывод отсюда осторожный: в таблице все водители-участники, а не виновники, и неизвестно, сколько водителей с каким стажем ездит по дорогам, поэтому сказать по этим числам, кто опаснее, нельзя. Строка «Без пострадавших» — один водитель, малая группа.

Третий вопрос, пешеходы днём и ночью: маска гибели, сгруппированная по маске тёмного времени.

In [ ]:
pedestrians = both[both['role'] == 'Пешеход']                            # пешеходы с их ДТП
dark = pedestrians['light'].str.contains('темное')                       # маска: тёмное время
(pedestrians['health'] == 'Погиб').groupby(dark).agg(['size', 'mean']).round(3)   # пешеходы и доля погибших

В светлое время и в сумерки погибает один сбитый пешеход из двенадцати, в тёмное время почти каждый четвёртый. Доля 0,237 уже встречалась у ночных часов, и это совпадение: там считались ДТП, здесь люди.

Объединённая таблица сохраняется в файл: для неё индекс не нужен, параметр `index=False`.

In [ ]:
both.to_csv('dtp-bryansk-joined.csv', index=False)                       # объединённая таблица в файл

Памятка проверки данных, шесть пунктов, по которым сегодня прошли таблицы ДТП и по которым стоит пройти свой датасет:

1. Типы: что прочиталось текстом, что стало дробным из-за пропусков.
2. Даты: переведены ли в настоящие даты, какой период они покрывают.
3. Пропуски: сколько, в каких столбцах, «не применимо» или «неизвестно».
4. Коды и невозможные значения: 99 лет стажа, год выпуска 1.
5. Повторы: ошибка или разные объекты с одинаковыми признаками.
6. Ключи: что однозначно называет строку и чем таблицы связаны между собой.

## Итоги

Тип столбца виден в `dtypes` и меняется методом `astype`; целые числа с пропусками хранит тип `Int64`. Текст становится датой функцией `pd.to_datetime`, после чего работают свойства `dt`, сравнение с датой-строкой и разность дат. Пропуски считаются через `isna`, заполняются `fillna`, строки с ними удаляются `dropna` с параметром `subset`; прежде чем удалять, надо понять, чьи это пропуски. Служебные коды заменяются пропусками методом `replace`, повторы находит `duplicated`, и не всякий повтор ошибка. Таблицы сопоставляются по ключу методом `merge` и складываются друг под другом функцией `pd.concat`.

Переходите к упражнениям: откройте `exercises/seminar-06-tasks.ipynb`, формулировки в `exercises/seminar-06-tasks.md`; данные там по Московской области. Следующее занятие — контрольная работа № 1 по NumPy и pandas: правила и пустой ноутбук в `notebooks/seminar-07-test-1.ipynb`, подготовка — открытый тест Moodle «КР 1. Демовариант». Из сегодняшнего материала в контрольную входят даты и пропуски.